In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import root_mean_squared_error
import os
import tarfile
import urllib.request

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

# Run the function
fetch_housing_data()

import pandas as pd

housing = pd.read_csv("datasets/housing/housing.csv")
housing = housing.rename(columns={'median_house_value' : 'price'})

## MY Model

In [ ]:
x = housing[:]
z = x.pop('ocean_proximity')
y = x.pop('price')

x.head()
y = y.fillna(y.median())

num_housing = housing.select_dtypes(include=['number'])
num_housing = num_housing.fillna(num_housing.median())

corr = num_housing.corr()
corr['price'].sort_values(ascending=False)



def score(X , Y, modelc, **kwargs):
    x_train,x_test, y_train,   y_test = train_test_split(X,Y, random_state=42,test_size=0.2)
    model = modelc(**kwargs)
    model.fit(x_train , y_train)

    
    preds = model.predict(x_test)
    Mae = root_mean_squared_error(preds, y_test)
    
    print(f' MAE : {Mae}')
    return model.score(x_test, y_test)*100, model.score(x_train , y_train) *100

 

score( x, y ,  RandomForestRegressor)

## Hands on ML model

In [ ]:



housing['rooms/house'] = housing['total_rooms']/ housing['households']
housing['bed/rooms'] = housing['total_bedrooms'] / housing['total_rooms']
housing['pop/house'] = housing['population'] / housing['households']




housing['ocean_proximity'].value_counts()
import numpy as np

def tts(data , test_ratio):
    shuffled_indices = np.random.permutation(len(data))
    split = int(len(data)* test_ratio)
    testing = shuffled_indices[:split]
    training = shuffled_indices[split:]

    return data.iloc[training] , data.iloc[testing]

train_size , test_size  = tts(housing , 0.2)
print(f'train size: {len(train_size)}  ,  test_size:  {len(test_size)}')
import hashlib

def test_size_check(identifier , test_ratio , hash):
    return hash(np.int64(identifier)).digest()[-1] < 256 * test_ratio

def split_train_test_by_id(data , test_ratio , id_col , hash = hashlib.md5):
    ids = data[id_col]
    in_test_set = ids.apply(lambda id_: test_size_check(id_ , test_ratio , hash))
    return data.loc[~in_test_set] , data.loc[in_test_set]

housing_id = housing.reset_index()
train_size, test_size = split_train_test_by_id(housing_id, 0.2, 'index')
housing_id['id'] = housing_id.longitude * 1000 + housing_id.latitude
train_size, test_size = split_train_test_by_id(housing_id, 0.2, 'id')
housing_id.head()
train_set , test_set = train_test_split(housing, random_state=42, test_size=0.2)

housing['income_cat'] = np.ceil(housing['price'])/1.5
housing['income_cat'].where(housing['income_cat'] < 5, 5.0 , inplace=True)


from sklearn.model_selection import StratifiedShuffleSplit


housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

from sklearn.model_selection import StratifiedShuffleSplit




# 1. Ensure you are using the FULL dataset here
# 2. Define the split
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 3. Perform the split
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# 4. NOW separate your features and labels to fix that "perfect" MSE from before
# Drop the 'price' from training features so the model can't "cheat"

# Remove 'income_cat' so it doesn't interfere with the math
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)





housing.plot(kind='scatter' , x='longitude' , y='latitude', alpha = 0.1)
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
 s=housing["population"]/100, label="population", figsize=(6,4),
 c="price", cmap=plt.get_cmap("jet"), colorbar=True,
)
plt.legend()
corr = housing.corr(numeric_only=True)
print(corr["price"].sort_values(ascending=False))


from pandas.plotting import scatter_matrix
att = ['price', 'median_income', 'total_rooms', 'housing_median_age']

scatter_matrix(housing[att], figsize=(12,8))

from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy = 'median')

housing_numerical = housing.drop('ocean_proximity', axis=1)

housing_numerical = housing.select_dtypes(include=['number'])
imp.fit(housing_numerical)

X = imp.transform(housing_numerical)
pd.DataFrame(X, columns=housing_numerical.columns).head()

housing_cat = housing['ocean_proximity']

factorise_housing_cat , housing_cat = housing_cat.factorize()
factorise_housing_cat[:10]


from sklearn.preprocessing import OneHotEncoder

one = OneHotEncoder()
encode_housing = one.fit_transform(factorise_housing_cat.reshape(-1,1))

encode_housing.toarray()


from sklearn.base import TransformerMixin, BaseEstimator

household_ix, rooms_ix, population_ix, bedrooms_ix = 6,2,5,4

class CombinedAttributesAdder(BaseEstimator, TransformerMixin ):

    def __init__(self, add_bedrooms_per_rooms=True):
        self.add_bedrooms_per_rooms = add_bedrooms_per_rooms

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:,household_ix]
        pop_per_household = X[:, population_ix] / X[:,household_ix]

        if self.add_bedrooms_per_rooms:
            bedrooms_per_household = X[:,bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household,pop_per_household ,bedrooms_per_household ]
        else :
            return   np.c_[X, rooms_per_household, pop_per_household]


attributes_adder = CombinedAttributesAdder(add_bedrooms_per_rooms=False)
extra_housing_attributes = attributes_adder.transform(housing.values)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


num_pipeline = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')),
    ('attri_adder_pipeline' , CombinedAttributesAdder()),
    ('StandardScaler' , StandardScaler())


])
class DataFrameSelector:

    def __init__(self, attributes_adder):
        self.attributes_names = attributes_adder

    def fit(self,X,y=None):
        return self
    
    def transform(self, X):
        return X[self.attributes_names].values


numerical_attributes = list(housing_numerical)
categorical_attributes = ['ocean_proximity']


housing_num_tr = num_pipeline.fit_transform(housing_numerical)


from sklearn.compose import ColumnTransformer

full_pipeline = ColumnTransformer([
    ('num_pipeline' , num_pipeline, numerical_attributes),
    ('cat_pipeline' ,OneHotEncoder(), categorical_attributes)
])

housing_prepared = full_pipeline.fit_transform(housing)


